In [3]:
from pathlib import Path
import pandas as pd
import pdfplumber
from pathlib import Path

year = "2025"
quarter = "q1"

pic_path = f"pic/{year}/{quarter}/"

sizes = ["small", "large", "superlarge", "medium"]

size = "small"
pdf_path = f"data/{year}/{year}_{quarter}_{size}.pdf"

Path(pic_path).mkdir(parents=True, exist_ok=True)

In [ ]:

all_rows = []
header = None

with pdfplumber.open(pdf_path) as pdf:
	for page_number, page in enumerate(pdf.pages, start=1):
		tables = page.extract_tables({
			"vertical_strategy": "text",      # 基于文本垂直对齐分列
			"horizontal_strategy": "text",    # 基于文本水平对齐分行
		})
		for table in tables:
			for row_idx, row in enumerate(table):
				# Capture the header from the very first table on page 1
				if header is None and page_number == 1 and row_idx == 0:
					header = row
					all_rows.append(row)
				else:
					# Skip repeated headers if they reappear at the top of subsequent pages
					if row == header:
						continue
					all_rows.append(row)
     
if all_rows:
	df = pd.DataFrame(all_rows[1:], columns=all_rows[0])
else:
	df = pd.DataFrame()

df.dropna(thresh=3)
print(df)
df.to_csv(f"{year}_{quarter}_{size}.csv", index=False, encoding="utf_8_sig")
   

               （电梯维保业务承     接量 50   至 500     台（含   ），共 1    00 家）          \
0                                                                            
1                            电梯维护  保养基本情况            *电梯困  人应急救援情况           
2    信用评价                                                                    
3                                                                            
4                电梯维保单位  人均维保台量 平   均维保时长    定期检验  电梯救援工单   虚假完工工单    平均救援   
..    ...           ...       ...     ...     ...     ...      ...     ...   
199    98  上海盛菱电梯工程有限公司     27.43  34分02秒  75.71%       2        0   7分32秒   
200                                                                          
201    99    上海亦快电梯有限公司     27.29  35分56秒  70.00%       3        1  20分18秒   
202                                                                          
203   100    曼隆蒂升电梯有限公司     26.57  37分13秒    100%       2        0  14分13秒   

              
0             
1    *受到电梯相关  
2             
3  

In [26]:
for pdf_path in pdf_files:
	with pdfplumber.open(pdf_path) as pdf:
		for page_number, page in enumerate(pdf.pages, start=1):
			text = page.extract_text()
			print(f"PDF: {pdf_path}, Page: {page_number}, Text: {text}")

PDF: data\2026\2026_q2_large.pdf, Page: 1, Text: 
PDF: data\2026\2026_q2_large.pdf, Page: 2, Text: 
PDF: data\2026\2026_q2_large.pdf, Page: 3, Text: 
PDF: data\2026\2026_q2_medium.pdf, Page: 1, Text: 
PDF: data\2026\2026_q2_medium.pdf, Page: 2, Text: 
PDF: data\2026\2026_q2_medium.pdf, Page: 3, Text: 
PDF: data\2026\2026_q2_medium.pdf, Page: 4, Text: 
PDF: data\2026\2026_q2_medium.pdf, Page: 5, Text: 
PDF: data\2026\2026_q2_medium.pdf, Page: 6, Text: 
PDF: data\2026\2026_q2_small.pdf, Page: 1, Text: 
PDF: data\2026\2026_q2_small.pdf, Page: 2, Text: 
PDF: data\2026\2026_q2_small.pdf, Page: 3, Text: 
PDF: data\2026\2026_q2_small.pdf, Page: 4, Text: 
PDF: data\2026\2026_q2_small.pdf, Page: 5, Text: 
PDF: data\2026\2026_q2_superlarge.pdf, Page: 1, Text: 


In [ ]:
dataframes = []

def unique_headers(row):
	headers = []
	counts = {}

	for index, value in enumerate(row):
		name = str(value).strip() if value is not None else ""
		name = name or f"column_{index + 1}"

		counts[name] = counts.get(name, 0) + 1
		if counts[name] > 1:
			name = f"{name}_{counts[name]}"

		headers.append(name)

	return headers

for pdf_path in pdf_files:
	with pdfplumber.open(pdf_path) as pdf:
		for page_number, page in enumerate(pdf.pages, start=1):
			for table_number, table in enumerate(page.extract_tables(), start=1):
				rows = [
					[cell.strip() if isinstance(cell, str) else cell for cell in row]
					for row in table
					if row and any(cell is not None and str(cell).strip() for cell in row)
				]

				if not rows:
					continue

				width = max(len(row) for row in rows)
				rows = [row + [None] * (width - len(row)) for row in rows]

				df = pd.DataFrame(rows[1:], columns=unique_headers(rows[0]))
				df.insert(0, "table_number", table_number)
				df.insert(0, "page_number", page_number)
				df.insert(0, "source_file", str(pdf_path))
				dataframes.append(df)

merged_df = (
	pd.concat(dataframes, ignore_index=True, sort=False)
	if dataframes
	else pd.DataFrame()
)

print(f"Found {len(pdf_files)} PDF file(s) and extracted {len(dataframes)} table(s).")
merged_df